# EDA — Presuntos Suicidios Colombia 2015-2024

Fuentes (en `data/bronze/`):
- `suicidios_raw.csv`: base del INMLCF descargada por la API (se genera al correr `python main.py`).
- Proyecciones de población departamental del DANE (2005-2017 y 2018-2050).

Se lee todo con `dtype=str` y **no se modifica nada**: el EDA solo detecta, cuantifica e interpreta. Los hallazgos se convierten después en funciones `limpiar_<fuente>` en `src/transform/clean_data.py`.

> Nota: en esta base el dato faltante viene como la categoría "Sin información", no como nulo; hay que contarla aparte.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
SIN_INFO = 'Sin información'

# Parte A — Base de suicidios (INMLCF)

## 1. Carga, tamaño y qué representa cada fila

In [ ]:
df = pd.read_csv('../data/bronze/suicidios_raw.csv', dtype=str)
print(df.shape)  # esperado: 26.558 filas
df.head()

## 2. Tipos de datos

In [ ]:
df.info()

## 3. Perfil de calidad (nulos, 'Sin información' y únicos)

In [ ]:
perfil = pd.DataFrame({
    'nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(2),
    'sin_info': df.apply(lambda c: c.str.strip().str.lower().eq(SIN_INFO.lower()).sum()),
    'unicos': df.nunique(),
})
perfil['pct_sin_info'] = (perfil['sin_info'] / len(df) * 100).round(2)
perfil.sort_values('pct_sin_info', ascending=False)

## 4. Duplicados: filas idénticas y llaves repetidas

In [ ]:
print('Filas idénticas:', df.duplicated().sum())
print('Filas idénticas sin contar id:', df.drop(columns='id').duplicated().sum())
print('Llaves repetidas (id):', df['id'].duplicated().sum())

## 5. Cardinalidad y categorías
No confiar en `head()`/`tail()`: las variantes (mayúsculas, tildes, redacción por año) suelen estar en el medio.

In [ ]:
for col in df.columns:
    if df[col].nunique() < 60:
        print(f'--- {col} ---')
        print(df[col].value_counts(dropna=False), '\n')

## 6. Formatos
Año, mes, día, códigos DANE (longitud y ceros a la izquierda).

In [ ]:
for col in ['a_o_del_hecho', 'codigo_dane_departamento', 'codigo_dane_municipio']:
    print(col, df[col].str.len().value_counts().to_dict())

## 7. Variables por año
Revisar si las categorías cambian de escritura según el año.

In [ ]:
# Ejemplo: pd.crosstab(df['mecanismo_causal_de_la_lesion_fatal'], df['a_o_del_hecho'])

## 8. Conteos de referencia

In [ ]:
df['a_o_del_hecho'].value_counts().sort_index()

# Parte B — Proyecciones de población DANE

Los archivos del DANE traen filas de título antes del encabezado. Aquí se identifica cuántas son para fijar `skiprows` en `config/config.yaml`.

In [ ]:
for archivo in ['DCD-area-proypoblacion-dep-2005-2017_VP.xlsx', 'PPED-AreaDep-2018-2050_VP.xlsx']:
    crudo = pd.read_excel(f'../data/bronze/{archivo}', header=None, dtype=str)
    print(archivo, crudo.shape)
    display(crudo.head(15))

In [ ]:
# Después de fijar skiprows:
# pob = pd.read_excel('../data/bronze/PPED-AreaDep-2018-2050_VP.xlsx', skiprows=TODO, dtype=str)
# pob.columns.tolist(), pob['ÁREA GEOGRÁFICA'].value_counts()

## Conclusiones
- TODO: hallazgo → decisión de limpieza (y su justificación)